# Draft.AI — Training Pipeline v2

| Component | Detail |
|---|---|
| **Models** | LightGBM + CatBoost Ensemble |
| **Calibration** | Isotonic Regression |
| **Explainability** | SHAP TreeExplainer |
| **Data** | Oracle's Elixir 2020–2026 + Solo Queue (league_data.csv) |
| **Split** | 80/20 stratified per year |
| **Features** | Champion encodings, matchup WR, synergy WR, role fit, patch, context |
| **Target** | Blue-side win probability |

In [ ]:
!pip install pandas numpy lightgbm catboost scikit-learn shap joblib

In [ ]:
import sys, os
sys.stdout.reconfigure(encoding='utf-8')

import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
from catboost import CatBoostClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, brier_score_loss
from sklearn.preprocessing import LabelEncoder
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded')

## 1. Data Loading — Oracle's Elixir

Load 2020–2026 Oracle's Elixir CSVs. Each game = 12 rows (5 players × 2 sides + 2 team summaries). Skips missing years gracefully.

In [ ]:
DATA_DIR = '../Dataset'
YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

dfs = []
for year in YEARS:
    path = os.path.join(DATA_DIR, f'{year}_LoL_esports_match_data_from_OraclesElixir.csv')
    if not os.path.exists(path):
        print(f'⚠  {year} not found, skipping')
        continue
    df = pd.read_csv(path, low_memory=False)
    print(f'✅ {year}: {df["gameid"].nunique():,} games')
    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)
print(f'\n📊 Total: {raw_df["gameid"].nunique():,} unique games')

## 2. Data Wrangling — Oracle's Elixir

1. Keep only `datacompleteness == 'complete'` rows
2. Filter to player rows (position ≠ 'team') in valid roles
3. Remove games with duplicate role assignments
4. Pivot: 10 player rows → 1 game row with champion per role/side
5. Merge blue-side context (result, patch, league, playoffs, year)

In [ ]:
raw_df = raw_df[raw_df['datacompleteness'] == 'complete'].copy()

player_df = raw_df[raw_df['position'] != 'team'].copy()
player_df['position'] = player_df['position'].str.lower()
player_df = player_df[player_df['position'].isin(['top', 'jng', 'mid', 'bot', 'sup'])]
player_df = player_df.dropna(subset=['champion'])

player_df['role_side'] = player_df['side'].str.lower() + '_' + player_df['position']
dupes = player_df.groupby(['gameid', 'role_side']).size()
dupe_games = dupes[dupes > 1].reset_index()['gameid'].unique()
print(f'Games with duplicate role assignments (removed): {len(dupe_games)}')
player_df = player_df[~player_df['gameid'].isin(dupe_games)]

draft_pivot = player_df.pivot_table(
    index='gameid',
    columns='role_side',
    values='champion',
    aggfunc='first'
).reset_index()

blue_context = player_df[player_df['side'] == 'Blue'].groupby('gameid').agg({
    'result': 'first',
    'patch': 'first',
    'league': 'first',
    'playoffs': 'first',
    'year': 'first'
}).reset_index()

games_df = draft_pivot.merge(blue_context, on='gameid', how='inner')

ROLE_COLS = [
    'blue_top', 'blue_jng', 'blue_mid', 'blue_bot', 'blue_sup',
    'red_top',  'red_jng',  'red_mid',  'red_bot',  'red_sup'
]

games_df = games_df.dropna(subset=ROLE_COLS)
games_df['result'] = games_df['result'].astype(int)
games_df['playoffs'] = pd.to_numeric(games_df['playoffs'], errors='coerce').fillna(0).astype(int)

print(f'✅ OraclesElixir dataset: {len(games_df):,} games')
print(f'   Blue side win rate: {games_df["result"].mean():.4f}')
print(f'   Years: {sorted(games_df["year"].unique())}')
games_df.head()

## 2b. Data Loading — Solo Queue (league_data.csv)

Riot API ranked solo queue data. Different format: one row per player, `team_id=100` = blue, `team_position` = role, `win=TRUE/FALSE`.

Wrangled into same shape as OraclesElixir and concatenated. `is_tier1=0` for all solo queue games (not pro leagues).

> **Note:** Solo queue and pro play have different dynamics. Solo queue adds volume and broader champion pool coverage.

In [ ]:
POS_MAP = {'TOP': 'top', 'JUNGLE': 'jng', 'MIDDLE': 'mid', 'BOTTOM': 'bot', 'UTILITY': 'sup'}
league_csv = os.path.join(DATA_DIR, 'league_data.csv')

if os.path.exists(league_csv):
    print('📂 Loading league_data.csv...')
    lol_raw = pd.read_csv(league_csv, low_memory=False, encoding='latin-1')

    lol_raw['position'] = lol_raw['team_position'].map(POS_MAP)
    lol_raw = lol_raw[lol_raw['position'].notna()]
    lol_raw['side'] = lol_raw['team_id'].map({100: 'Blue', 200: 'Red'})
    lol_raw['role_side'] = lol_raw['side'].str.lower() + '_' + lol_raw['position']
    lol_raw['win_bool'] = lol_raw['win'].astype(str).str.upper() == 'TRUE'
    lol_raw['patch'] = lol_raw['game_version'].apply(
        lambda v: '.'.join(str(v).split('.')[:2]) if pd.notna(v) else '0.0'
    )
    lol_raw['year'] = pd.to_datetime(lol_raw['game_start_utc'], errors='coerce').dt.year

    dupes_lol = lol_raw.groupby(['game_id', 'role_side']).size()
    dupe_games_lol = dupes_lol[dupes_lol > 1].reset_index()['game_id'].unique()
    print(f'   Games with duplicate roles (removed): {len(dupe_games_lol)}')
    lol_raw = lol_raw[~lol_raw['game_id'].isin(dupe_games_lol)]

    lol_pivot = lol_raw.pivot_table(
        index='game_id',
        columns='role_side',
        values='champion_name',
        aggfunc='first'
    ).reset_index().rename(columns={'game_id': 'gameid'})

    blue_lol = lol_raw[lol_raw['side'] == 'Blue'].groupby('game_id').agg(
        result=('win_bool', lambda x: int(x.iloc[0])),
        patch=('patch', 'first'),
        league=('queue_id', lambda x: 'SoloQ'),
        playoffs=('queue_id', lambda x: 0),
        year=('year', 'first')
    ).reset_index().rename(columns={'game_id': 'gameid'})

    games_lol = lol_pivot.merge(blue_lol, on='gameid', how='inner')
    games_lol = games_lol.dropna(subset=ROLE_COLS)
    games_lol['result'] = games_lol['result'].astype(int)
    games_lol['playoffs'] = 0

    print(f'   ✅ league_data.csv: {len(games_lol):,} games')
    print(f'      Blue WR: {games_lol["result"].mean():.4f}')
    print(f'      Years: {sorted(games_lol["year"].dropna().unique())}')

    games_df = pd.concat([games_df, games_lol], ignore_index=True)
    print(f'\n📊 Combined dataset: {len(games_df):,} games total')
else:
    print('⚠  league_data.csv not found in Dataset/, skipping')

print(f'\n✅ Final dataset: {len(games_df):,} games')
print(f'   Blue side win rate: {games_df["result"].mean():.4f}')
print(f'   Unique champions: {pd.concat([games_df[c] for c in ROLE_COLS]).nunique()}')

## 3. Train / Test Split

**80/20 stratified by year** — each year contributes exactly 80% to train and 20% to test. `stratify=year` prevents any year from being over/under-represented in either split.

In [ ]:
train_df, test_df = train_test_split(
    games_df,
    test_size=0.20,
    random_state=42,
    stratify=games_df['year']
)

train_mask = games_df.index.isin(train_df.index)
test_mask  = games_df.index.isin(test_df.index)

train_df = train_df.copy()
test_df  = test_df.copy()

base_rate = float(train_df['result'].mean())
print(f'📊 Train: {len(train_df):,} | Test: {len(test_df):,}')
print(f'   Train year distribution: {train_df["year"].value_counts().sort_index().to_dict()}')
print(f'   Test year distribution:  {test_df["year"].value_counts().sort_index().to_dict()}')
print(f'   Base rate (blue WR): {base_rate:.4f}')

## 4. Feature Engineering

| Feature Group | Count | Description |
|---|---|---|
| Champion encodings | 10 | Label-encoded champion ID per role slot |
| Matchup win rates | 5 | Same-role head-to-head smoothed WR |
| Synergy pairs | 6 | Bot+Sup, Mid+Jng, Top+Jng (both sides) |
| Role fit scores | 10 | Fraction of games champion played in this role |
| Context | 3 | patch_numeric, playoffs, is_tier1 |

All lookup tables computed on **train set only** to prevent leakage.

In [ ]:
SMOOTHING = 20
ROLES = ['top', 'jng', 'mid', 'bot', 'sup']

def parse_patch(p):
    try:
        parts = str(p).split('.')
        return int(parts[0]) * 100 + int(parts[1])
    except Exception:
        return 0

games_df['patch_numeric'] = games_df['patch'].apply(parse_patch)
print('✅ Patch numeric computed')

In [ ]:
print('🔧 Computing same-role matchup features...')
matchup_tables = {}
for role in ROLES:
    bc = f'blue_{role}'
    rc = f'red_{role}'
    grp = train_df.groupby([bc, rc])['result'].agg(['mean', 'count'])
    grp.columns = ['wr', 'count']
    grp = grp[grp['count'] >= 3]
    grp['smoothed'] = (grp['wr'] * grp['count'] + base_rate * SMOOTHING) / (grp['count'] + SMOOTHING)
    matchup_tables[role] = grp['smoothed'].to_dict()

for role in ROLES:
    bc = f'blue_{role}'
    rc = f'red_{role}'
    games_df[f'{role}_matchup_wr'] = [
        matchup_tables[role].get((b, r), base_rate)
        for b, r in zip(games_df[bc], games_df[rc])
    ]

MATCHUP_COLS = [f'{role}_matchup_wr' for role in ROLES]
print(f'   ✅ {len(MATCHUP_COLS)} matchup features added')

In [ ]:
print('🔧 Computing synergy pair features...')
SYNERGY_PAIRS = [
    ('blue', 'bot', 'sup'), ('blue', 'mid', 'jng'), ('blue', 'top', 'jng'),
    ('red',  'bot', 'sup'), ('red',  'mid', 'jng'), ('red',  'top', 'jng'),
]

synergy_tables = {}
for side, r1, r2 in SYNERGY_PAIRS:
    c1 = f'{side}_{r1}'
    c2 = f'{side}_{r2}'
    key = f'{side}_{r1}_{r2}_synergy_wr'
    if side == 'red':
        grp_vals = train_df.groupby([c1, c2])['result'].agg(wr=lambda x: 1.0 - x.mean(), count='count')
    else:
        grp_vals = train_df.groupby([c1, c2])['result'].agg(wr='mean', count='count')
    grp_vals = grp_vals[grp_vals['count'] >= 3]
    grp_vals['smoothed'] = (grp_vals['wr'] * grp_vals['count'] + base_rate * SMOOTHING) / (grp_vals['count'] + SMOOTHING)
    synergy_tables[key] = grp_vals['smoothed'].to_dict()
    games_df[key] = [synergy_tables[key].get((a, b), base_rate) for a, b in zip(games_df[c1], games_df[c2])]

SYNERGY_COLS = [f'{s}_{r1}_{r2}_synergy_wr' for s, r1, r2 in SYNERGY_PAIRS]
print(f'   ✅ {len(SYNERGY_COLS)} synergy features added')

In [ ]:
le = LabelEncoder()
all_champions = pd.concat([games_df[c] for c in ROLE_COLS]).dropna().unique()
le.fit(all_champions)

ENCODED_COLS = []
for col in ROLE_COLS:
    enc_col = f'{col}_enc'
    games_df[enc_col] = le.transform(games_df[col])
    ENCODED_COLS.append(enc_col)
print(f'✅ {len(le.classes_)} champions label-encoded')

TIER1_LEAGUES = {'LCK', 'LPL', 'LEC', 'LCS', 'MSI', 'Worlds'}
games_df['is_tier1'] = games_df['league'].isin(TIER1_LEAGUES).astype(int)
print('✅ Tier-1 league flag added (SoloQ games = 0)')

print('🔧 Computing role fit scores...')
role_fit_table = {}
for role in ROLES:
    appearances = pd.concat([train_df[f'blue_{role}'], train_df[f'red_{role}']]).dropna()
    for champ in appearances:
        role_fit_table.setdefault(champ, {})
        role_fit_table[champ][role] = role_fit_table[champ].get(role, 0) + 1

for champ in role_fit_table:
    total = sum(role_fit_table[champ].values())
    if total > 0:
        for role in role_fit_table[champ]:
            role_fit_table[champ][role] = round(role_fit_table[champ][role] / total, 4)

for col in ROLE_COLS:
    side, role = col.split('_', 1)
    games_df[f'{col}_role_fit'] = [
        role_fit_table.get(champ, {}).get(role, 0.2) for champ in games_df[col]
    ]

ROLE_FIT_COLS = [f'{col}_role_fit' for col in ROLE_COLS]
FEATURE_COLS = ENCODED_COLS + MATCHUP_COLS + SYNERGY_COLS + ROLE_FIT_COLS + ['patch_numeric', 'playoffs', 'is_tier1']
print(f'\n📐 Total features: {len(FEATURE_COLS)}')
print(f'   Encodings: {len(ENCODED_COLS)} | Matchup: {len(MATCHUP_COLS)} | Synergy: {len(SYNERGY_COLS)} | Role fit: {len(ROLE_FIT_COLS)} | Context: 3')

## 5. Prepare Train / Test Arrays

Recency weights: 2026 games weighted 5× more than 2020. Solo queue games get `year` from `game_start_utc`, so they get weighted by their year too.

In [ ]:
WEIGHT_MAP = {2020: 0.3, 2021: 0.4, 2022: 0.5, 2023: 0.8, 2024: 1.0, 2025: 1.2, 2026: 1.5}

X_train = games_df.loc[train_mask, FEATURE_COLS].astype(float)
X_test  = games_df.loc[test_mask,  FEATURE_COLS].astype(float)
y_train = games_df.loc[train_mask, 'result']
y_test  = games_df.loc[test_mask,  'result']
w_train = games_df.loc[train_mask, 'year'].map(WEIGHT_MAP).fillna(1.0)

X_fit, X_cal, y_fit, y_cal, w_fit, _ = train_test_split(
    X_train, y_train, w_train,
    test_size=0.20, random_state=42, stratify=y_train
)

print(f'X_fit:  {X_fit.shape}  (model training)')
print(f'X_cal:  {X_cal.shape}  (calibration)')
print(f'X_test: {X_test.shape} (final evaluation)')

## 6. Model 1 — LightGBM (Alone)

Gradient boosted decision trees by Microsoft. Leaf-wise growth — always splits the leaf with highest loss reduction first. Learns champion interactions through thousands of decision trees.

In [ ]:
print('🚀 Training LightGBM...')
lgb_params = {
    'objective': 'binary', 'metric': ['binary_logloss', 'auc'],
    'boosting_type': 'gbdt', 'num_leaves': 127, 'learning_rate': 0.03,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'min_child_samples': 30, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'verbose': -1, 'seed': 42
}

lgb_train_ds = lgb.Dataset(X_fit, label=y_fit, weight=w_fit, free_raw_data=False)
lgb_val_ds   = lgb.Dataset(X_cal, label=y_cal, reference=lgb_train_ds, free_raw_data=False)

lgb_model = lgb.train(
    lgb_params, lgb_train_ds, num_boost_round=2000,
    valid_sets=[lgb_val_ds], valid_names=['valid'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)

lgb_cal_pred  = lgb_model.predict(X_cal)
lgb_test_pred = lgb_model.predict(X_test)

# Calibrate LightGBM alone
lgb_calibrator = IsotonicRegression(out_of_bounds='clip')
lgb_calibrator.fit(lgb_cal_pred, y_cal)
lgb_cal_final = lgb_calibrator.transform(lgb_test_pred)
lgb_cal_final = np.clip(lgb_cal_final, 0.15, 0.85)

lgb_auc  = roc_auc_score(y_test, lgb_cal_final)
lgb_acc  = accuracy_score(y_test, (lgb_cal_final > 0.5).astype(int))
lgb_ll   = log_loss(y_test, lgb_cal_final)
lgb_bs   = brier_score_loss(y_test, lgb_cal_final)

print(f'\n📊 LightGBM (Calibrated) Results:')
print(f'   AUC:      {lgb_auc:.4f}')
print(f'   Accuracy: {lgb_acc:.4f}  ({lgb_acc*100:.1f}%)')
print(f'   Log Loss: {lgb_ll:.4f}')
print(f'   Brier:    {lgb_bs:.4f}')

## 7. Model 2 — CatBoost (Alone)

Gradient boosted decision trees by Yandex. Uses ordered target encoding for categorical features — encodes each champion using only games seen before it in a shuffled order, preventing target leakage. Learns different interaction patterns than LightGBM.

In [ ]:
print('🚀 Training CatBoost...')
cb_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.03, depth=7, l2_leaf_reg=3,
    random_seed=42, verbose=200, eval_metric='AUC', early_stopping_rounds=100,
)

cb_model.fit(X_fit, y_fit, sample_weight=w_fit.values, eval_set=(X_cal, y_cal), use_best_model=True)

cb_cal_pred  = cb_model.predict_proba(X_cal)[:, 1]
cb_test_pred = cb_model.predict_proba(X_test)[:, 1]

# Calibrate CatBoost alone
cb_calibrator = IsotonicRegression(out_of_bounds='clip')
cb_calibrator.fit(cb_cal_pred, y_cal)
cb_cal_final = cb_calibrator.transform(cb_test_pred)
cb_cal_final = np.clip(cb_cal_final, 0.15, 0.85)

cb_auc  = roc_auc_score(y_test, cb_cal_final)
cb_acc  = accuracy_score(y_test, (cb_cal_final > 0.5).astype(int))
cb_ll   = log_loss(y_test, cb_cal_final)
cb_bs   = brier_score_loss(y_test, cb_cal_final)

print(f'\n📊 CatBoost (Calibrated) Results:')
print(f'   AUC:      {cb_auc:.4f}')
print(f'   Accuracy: {cb_acc:.4f}  ({cb_acc*100:.1f}%)')
print(f'   Log Loss: {cb_ll:.4f}')
print(f'   Brier:    {cb_bs:.4f}')

## 8. Model 3 — Ensemble (LightGBM + CatBoost Together)

Average both models' raw probabilities before calibrating. Two models that learned differently → their errors don't perfectly overlap → averaging cancels out individual mistakes → better combined prediction.

In [ ]:
print('🔧 Building ensemble + calibrating...')

ens_cal  = (lgb_cal_pred  + cb_cal_pred)  / 2
ens_test = (lgb_test_pred + cb_test_pred) / 2

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(ens_cal, y_cal)
cal_pred_test = calibrator.transform(ens_test)
cal_pred_test = np.clip(cal_pred_test, 0.15, 0.85)

final_auc  = roc_auc_score(y_test, cal_pred_test)
final_acc  = accuracy_score(y_test, (cal_pred_test > 0.5).astype(int))
final_ll   = log_loss(y_test, cal_pred_test)
final_bs   = brier_score_loss(y_test, cal_pred_test)

print(f'\n📊 Ensemble (Calibrated) Results:')
print(f'   AUC:      {final_auc:.4f}')
print(f'   Accuracy: {final_acc:.4f}  ({final_acc*100:.1f}%)')
print(f'   Log Loss: {final_ll:.4f}')
print(f'   Brier:    {final_bs:.4f}')

## 9. Model Comparison — Why Ensemble Wins

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.calibration import calibration_curve

models = {
    'LightGBM':        {'preds': lgb_cal_final,  'auc': lgb_auc,   'acc': lgb_acc,   'll': lgb_ll,   'bs': lgb_bs,   'color': '#3b82f6'},
    'CatBoost':        {'preds': cb_cal_final,   'auc': cb_auc,    'acc': cb_acc,    'll': cb_ll,    'bs': cb_bs,    'color': '#10b981'},
    'Ensemble (Ours)': {'preds': cal_pred_test,  'auc': final_auc, 'acc': final_acc, 'll': final_ll, 'bs': final_bs, 'color': '#f59e0b'},
}

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.patch.set_facecolor('#0f0f23')

# --- Plot 1: AUC comparison ---
names  = list(models.keys())
aucs   = [m['auc'] for m in models.values()]
colors = [m['color'] for m in models.values()]
bars = axes[0].bar(names, aucs, color=colors, edgecolor='white', linewidth=0.8, width=0.5)
axes[0].set_ylim(min(aucs) - 0.005, max(aucs) + 0.005)
axes[0].set_title('AUC-ROC Comparison', fontweight='bold', color='white', fontsize=13)
axes[0].set_ylabel('AUC-ROC', color='white')
for bar, val in zip(bars, aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.0005,
                 f'{val:.4f}', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')

# --- Plot 2: Accuracy + Log Loss side by side ---
x = np.arange(len(names))
w = 0.35
accs = [m['acc'] for m in models.values()]
lls  = [m['ll']  for m in models.values()]
b1 = axes[1].bar(x - w/2, accs, w, color=colors, edgecolor='white', linewidth=0.8, label='Accuracy', alpha=0.9)
b2 = axes[1].bar(x + w/2, lls,  w, color=colors, edgecolor='white', linewidth=0.8, label='Log Loss', alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, color='white', fontsize=9)
axes[1].set_title('Accuracy vs Log Loss', fontweight='bold', color='white', fontsize=13)
axes[1].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
for bar, val in zip(b1, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.001,
                 f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)
for bar, val in zip(b2, lls):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.001,
                 f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)

# --- Plot 3: Calibration curves ---
axes[2].plot([0,1],[0,1], 'w--', alpha=0.4, label='Perfect calibration')
for name, m in models.items():
    prob_true, prob_pred = calibration_curve(y_test, m['preds'], n_bins=10, strategy='uniform')
    axes[2].plot(prob_pred, prob_true, marker='o', label=name, color=m['color'], linewidth=2)
axes[2].set_title('Calibration Curves', fontweight='bold', color='white', fontsize=13)
axes[2].set_xlabel('Mean Predicted Probability', color='white')
axes[2].set_ylabel('Fraction of Positives (Actual)', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)

for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('#444')
    ax.spines['left'].set_color('#444')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f23')
plt.show()

# --- Summary table ---
print('\n' + '='*65)
print(f'{"Model":<20} {"AUC":>8} {"Accuracy":>10} {"Log Loss":>10} {"Brier":>8}')
print('-'*65)
for name, m in models.items():
    marker = ' ✅' if name == 'Ensemble (Ours)' else ''
    print(f'{name:<20} {m["auc"]:>8.4f} {m["acc"]:>10.4f} {m["ll"]:>10.4f} {m["bs"]:>8.4f}{marker}')
print('='*65)

# --- Why ensemble wins ---
print('\n💡 WHY ENSEMBLE BEATS INDIVIDUAL MODELS:')
print(f'   LightGBM vs Ensemble AUC gain:  +{final_auc - lgb_auc:+.4f}')
print(f'   CatBoost vs Ensemble AUC gain:  +{final_auc - cb_auc:+.4f}')
print('''
   LightGBM and CatBoost learn differently:
   - LightGBM: leaf-wise tree growth, faster, captures deep interactions
   - CatBoost:  ordered target encoding, better on categorical features

   When one model is wrong, the other is often right.
   Averaging their predictions cancels out individual errors.
   This is called VARIANCE REDUCTION — the ensemble is more stable
   than either model alone across different drafts and patches.
''')

## 9b. SHAP Analysis

In [ ]:
print('🔧 Computing SHAP values on test set...')
FEATURE_DISPLAY = {
    'blue_top_enc': 'Blue Top Pick', 'blue_jng_enc': 'Blue Jungle Pick',
    'blue_mid_enc': 'Blue Mid Pick', 'blue_bot_enc': 'Blue Bot Pick',
    'blue_sup_enc': 'Blue Support Pick', 'red_top_enc': 'Red Top Pick',
    'red_jng_enc': 'Red Jungle Pick', 'red_mid_enc': 'Red Mid Pick',
    'red_bot_enc': 'Red Bot Pick', 'red_sup_enc': 'Red Support Pick',
    'top_matchup_wr': 'Top Lane Matchup', 'jng_matchup_wr': 'Jungle Matchup',
    'mid_matchup_wr': 'Mid Lane Matchup', 'bot_matchup_wr': 'Bot Lane Matchup',
    'sup_matchup_wr': 'Support Matchup',
    'blue_bot_sup_synergy_wr': 'Blue Bot-Sup Synergy',
    'blue_mid_jng_synergy_wr': 'Blue Mid-Jng Synergy',
    'blue_top_jng_synergy_wr': 'Blue Top-Jng Synergy',
    'red_bot_sup_synergy_wr': 'Red Bot-Sup Synergy',
    'red_mid_jng_synergy_wr': 'Red Mid-Jng Synergy',
    'red_top_jng_synergy_wr': 'Red Top-Jng Synergy',
    'patch_numeric': 'Patch / Meta Recency',
    'playoffs': 'Playoffs Game', 'is_tier1': 'Tier-1 League',
}

explainer = shap.TreeExplainer(lgb_model)
shap_sample = X_test.iloc[:2000] if len(X_test) > 2000 else X_test
shap_arr = explainer.shap_values(shap_sample)
if isinstance(shap_arr, list):
    shap_arr = shap_arr[1]

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = float(expected_value[1])
else:
    expected_value = float(expected_value)

mean_abs_shap = np.abs(shap_arr).mean(axis=0)
global_importance = sorted([
    {'feature': col, 'display': FEATURE_DISPLAY.get(col, col),
     'mean_abs_shap': round(float(mean_abs_shap[i]), 6),
     'normalized': round(float(mean_abs_shap[i] / mean_abs_shap.sum()), 4)}
    for i, col in enumerate(FEATURE_COLS)
], key=lambda x: x['mean_abs_shap'], reverse=True)

print(f'   Expected value (base): {expected_value:.4f}')
print(f'   Top 5 features by |SHAP|:')
for f in global_importance[:5]:
    print(f'     {f["display"]:<30}  {f["mean_abs_shap"]:.6f}  ({f["normalized"]*100:.1f}%)')

In [ ]:
print('   Computing per-champion SHAP values...')
champion_shap = {}

for i, col in enumerate(ENCODED_COLS):
    role_col = col.replace('_enc', '')
    col_shap = shap_arr[:, i]
    for champ in le.classes_:
        enc_val = int(le.transform([champ])[0])
        mask = shap_sample[col].values == enc_val
        if mask.sum() < 5:
            continue
        champion_shap.setdefault(champ, {})[role_col] = {
            'shap_value': round(float(col_shap[mask].mean()), 5),
            'count': int(mask.sum()),
        }

blue_champ_shap, red_champ_shap = [], []
for champ, slots in champion_shap.items():
    bd = {k: v for k, v in slots.items() if k.startswith('blue_')}
    rd = {k: v for k, v in slots.items() if k.startswith('red_')}
    if bd:
        best = max(bd.items(), key=lambda x: x[1]['shap_value'])
        blue_champ_shap.append({'champion': champ, 'role': best[0].replace('blue_','').upper(),
                                'shap_value': best[1]['shap_value'], 'count': best[1]['count']})
    if rd:
        best = max(rd.items(), key=lambda x: x[1]['shap_value'])
        red_champ_shap.append({'champion': champ, 'role': best[0].replace('red_','').upper(),
                               'shap_value': best[1]['shap_value'], 'count': best[1]['count']})

blue_champ_shap.sort(key=lambda x: x['shap_value'], reverse=True)
red_champ_shap.sort(key=lambda x: x['shap_value'], reverse=True)

print('   Top 5 blue-side champions by SHAP:')
for c in blue_champ_shap[:5]:
    print(f'     {c["champion"]:<20} {c["role"]:<8} SHAP={c["shap_value"]:+.5f}  n={c["count"]}')

shap_export = {
    'expected_value': round(expected_value, 4),
    'global_importance': global_importance,
    'top_blue_by_shap': blue_champ_shap[:20],
    'top_red_by_shap':  red_champ_shap[:20],
    'champion_shap':    champion_shap,
}
print('   ✅ SHAP analysis complete')

## 10. Export Model Artifacts

In [ ]:
print('📦 Exporting model artifacts...')

SMOOTHING_EXPORT = 20
train_games_df   = games_df[train_mask]
base_rate_export = float(train_games_df['result'].mean())

champion_values = {}
for role_col in ROLE_COLS:
    side, role = role_col.split('_', 1)
    champ_wr = train_games_df.groupby(role_col)['result'].agg(['mean', 'count'])
    champ_wr.columns = ['win_rate', 'games']
    champ_wr = champ_wr[champ_wr['games'] >= 5]
    for champ, row in champ_wr.iterrows():
        smoothed = (row['win_rate'] * row['games'] + base_rate_export * SMOOTHING_EXPORT) / (row['games'] + SMOOTHING_EXPORT)
        champion_values.setdefault(champ, {}).setdefault(side, {})[role] = {
            'win_rate': round(float(smoothed), 4), 'raw_win_rate': round(float(row['win_rate']), 4),
            'games': int(row['games']), 'value': round(float(smoothed - base_rate_export), 4)
        }

champion_stats = {}
blue_cols = [c for c in ROLE_COLS if c.startswith('blue_')]
red_cols  = [c for c in ROLE_COLS if c.startswith('red_')]

for champ in sorted(le.classes_):
    b_mask = train_games_df[blue_cols].eq(champ).any(axis=1)
    r_mask = train_games_df[red_cols].eq(champ).any(axis=1)
    b_count, r_count = int(b_mask.sum()), int(r_mask.sum())
    total = b_count + r_count
    if total == 0:
        continue
    b_wr = float(train_games_df.loc[b_mask, 'result'].mean()) if b_count > 0 else 0.5
    r_wr = 1.0 - float(train_games_df.loc[r_mask, 'result'].mean()) if r_count > 0 else 0.5
    champion_stats[champ] = {
        'total_picks': total, 'blue_picks': b_count, 'red_picks': r_count,
        'blue_win_rate': round(b_wr, 4), 'red_win_rate': round(r_wr, 4),
        'overall_win_rate': round((b_wr * b_count + r_wr * r_count) / total, 4)
    }

model_export = {
    'metadata': {
        'model_type': 'LightGBM + CatBoost Ensemble v2 (Isotonic Calibration)',
        'training_years': '2020-2026 (80% stratified per year)',
        'test_split': '20% stratified per year',
        'training_games': int(len(y_train)), 'test_games': int(len(y_test)),
        'test_auc': round(final_auc, 4), 'test_accuracy': round(final_acc, 4),
        'test_log_loss': round(final_ll, 4), 'test_brier_score': round(final_bs, 4),
        'base_blue_win_rate': round(base_rate_export, 4),
        'champion_count': len(le.classes_), 'features': FEATURE_COLS,
    },
    'champion_values': champion_values,
    'champion_stats': champion_stats,
    'label_encoder_classes': le.classes_.tolist()
}

json_path = '../src/data/draft_model.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(model_export, f, indent=2)
print(f'   ✅ JSON exported → {json_path}  ({os.path.getsize(json_path)/1024:.1f} KB)')

joblib.dump(lgb_model,  'lgb_model.pkl')
joblib.dump(cb_model,   'cb_model.pkl')
joblib.dump(calibrator, 'calibrated_ensemble.pkl')
joblib.dump(le,         'label_encoder.pkl')
print('   ✅ Model pkl files saved')

meta = {
    'feature_cols': FEATURE_COLS, 'encoded_cols': ENCODED_COLS,
    'matchup_cols': MATCHUP_COLS, 'synergy_cols': SYNERGY_COLS, 'role_fit_cols': ROLE_FIT_COLS,
    'matchup_tables': {role: {str(k): float(v) for k, v in tbl.items()} for role, tbl in matchup_tables.items()},
    'synergy_tables': {key: {str(k): float(v) for k, v in tbl.items()} for key, tbl in synergy_tables.items()},
    'role_fit_table': role_fit_table,
    'base_rate': base_rate_export, 'synergy_pair_defs': SYNERGY_PAIRS, 'shap': shap_export,
}
with open('model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f)
print('   ✅ model_meta.json saved')
print('\n✅ Training complete!')
print(f'\n🎯 ACCURACY: {final_acc*100:.1f}%  |  AUC: {final_auc:.4f}  |  LOG-LOSS: {final_ll:.4f}  |  BRIER: {final_bs:.4f}')